# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")

## 2. SEML Pipeline

## 2.1 Response Generator

In [4]:
exp_id = "09-17-1-test"

## 2.2 Pipeline

### Set up the experiment

In [3]:
import logging
import os
from dotenv import load_dotenv
from huggingface_hub import login
import torch

from src.evaluations.evaluate_reliability import evaluate_reliability

# Set up logging
logger = logging.getLogger("quant_logger")
logger.setLevel(logging.INFO)

# Set up cache paths
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = os.path.join(CACHE_PATH, "hub")

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")

print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH
os.environ["TRANSFORMERS_CACHE"] = CACHE_PATH

torch.hub.set_dir(CACHE_PATH)

# Empty the cache
with torch.no_grad():
    torch.cuda.empty_cache()

# HuggingFace authentication
load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')
if huggingface_token is None:
    raise ValueError(
        f"Please set the HUGGINGFACE_TOKEN environment variable. "
        f"Looking in {os.path.join(os.getcwd(), '.env')}"
    )
else:
    print("Hugging Face token loaded successfully.")
login(token=huggingface_token, add_to_git_credential=True)

def run_evaluate(
    # Exp ID
    exp_id: str,
    save_excel: bool = True,
    num_excel_rows: int = 20,
    
    # Reliability dataset parameters
    seed=123,
    max_new_tokens=25,
    temperature=0.1,
    use_beam_search=False,
    strategy="Direct Completion",
    dataset_name="",
    typo_type="none",
    typo_intensity=0,
    n_repeats=10,
    n_beams=5,
    max_entries=None,
    
    # Model parameters
    seed_model=123,
    model_name="",
    model_path="",
    device="cuda",
    cache_path=CACHE_PATH
):
    ##################
    ## Print config ##
    ##################
    print("Received the following configuration:")
    print(f"  Seed: {seed}")
    print(f"  Max new tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use beam search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset name: {dataset_name}")
    print(f"  Number of repeats: {n_repeats}")
    print(f"  Number of beams: {n_beams}")
    print(f"  Max entries: {max_entries}")
    print(f"  Seed model: {seed_model}")
    print(f"  Model name: {model_name}")
    print(f"  Model path: {model_path}")
    print(f"  Device: {device}")

    results = evaluate_reliability(
        exp_id=exp_id,
        model_name=model_name,
        dataset_name=dataset_name,
        typo_type=typo_type,
        typo_intensity=typo_intensity,
        strategy=strategy,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        n_repeats=n_repeats,
        n_beams=n_beams,
        max_entries=max_entries,
        save_excel=save_excel,
        num_excel_rows=num_excel_rows,
        cache_dir=cache_path
    )

    return results

# if __name__ == "__main__":
#     # Example usage
#     result = run_evaluate(
#         exp_id="test-run",
#         model_name="Llama-3-8B",
#         dataset_name="P17",
#         taxonomy_type="0",
#         strategy="Direct Completion",
#         max_entries=20,
#     )http://localhost:8008/tree?token=1f333809e711c109b6a50292336de304eefc47b99b6073c9
#     print(result)

### Run configurations

In [4]:
import itertools

# Fixed parameters
fixed_params = {
    'exp_id': "typo-test-09-29",
    'save_excel': True,
    'num_excel_rows': 20,
    'device': 'cuda',
    'seed': 42,
    'n_repeats': 10,
    'n_beams': 5,
    'max_entries': 20
}

# Grid parameters
grid_params = {
    'model_name': ['Llama-3-8B'],
    'max_new_tokens': [25],
    'temperature': [0.1],
    'use_beam_search': [False],
    'strategy': ["Direct Completion"],
    'dataset_name': ['P17'],
    'typo_type': [
        "none",
        "char_insertion",
        "char_deletion",
        "char_replacement",
        "char_repetition",
        "char_swapping",
        "word_CMW",
        "char_LCC",
        "word_synonym",
        "char_insert_noise",
        "word_repeat",
        "char_substitution",
        "word_emoji",
        "word_internet_slang",
        "word_phrase_translation",
        "word_context_aware_insertion",
        "word_remove_punctuation",
        "word_keyword_only",
        "word_taxonomy"
    ],
    'typo_intensity': [3]
}

# Generate all combinations for grid search
grid_combinations = list(itertools.product(
    grid_params['model_name'],
    grid_params['max_new_tokens'],
    grid_params['temperature'],
    grid_params['use_beam_search'],
    grid_params['strategy'],
    grid_params['dataset_name'],
    grid_params['typo_type'],
    grid_params['typo_intensity']
))

# Run the evaluate function for all combinations
results = []
max_combinations = 10000  # Set to a lower number for testing purposes
for i, combination in enumerate(grid_combinations):
    if i >= max_combinations:
        break
    model_name, max_new_tokens, temperature, use_beam_search, strategy, dataset_name, typo_type, typo_intensity = combination

    # Print current combination details
    print(f"Running combination {i+1}/{len(grid_combinations)}")
    print(f"  Model Name: {model_name}")
    print(f"  Max New Tokens: {max_new_tokens}")
    print(f"  Temperature: {temperature}")
    print(f"  Use Beam Search: {use_beam_search}")
    print(f"  Strategy: {strategy}")
    print(f"  Dataset Name: {dataset_name}")
    print(f"  Typo Type: {typo_type}")
    print(f"  Typo Intensity: {typo_intensity}")

    result = run_evaluate(
        exp_id=fixed_params['exp_id'],
        save_excel=fixed_params['save_excel'],
        num_excel_rows=fixed_params['num_excel_rows'],
        seed=fixed_params['seed'],
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_beam_search=use_beam_search,
        strategy=strategy,
        dataset_name=dataset_name,
        typo_type=typo_type,
        typo_intensity=typo_intensity,
        n_repeats=fixed_params['n_repeats'],
        n_beams=fixed_params['n_beams'],
        max_entries=fixed_params['max_entries'],
        model_name=model_name,
        device=fixed_params['device']
    )

    # Append result with parameter details
    results.append({
        'result': result,
        'parameters': {
            'model_name': model_name,
            'max_new_tokens': max_new_tokens,
            'temperature': temperature,
            'use_beam_search': use_beam_search,
            'strategy': strategy,
            'dataset_name': dataset_name,
            'typo_type': typo_type,
            'typo_intensity': typo_intensity
        }
    })

# Print or process the results as needed
for result in results:
    print(f"Parameters: {result['parameters']}")
    print(f"Results: {result['result']}")
    print("---")

## 3. Plot Results

In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fpdf import FPDF

def create_and_save_figures(df, plots_dir, model_name, strategy, max_new_tokens, temperature):
    plots = []
    metrics = ['Accuracy', 'AUCPR_sem']

    # Function to create a box plot
    def create_box_plot(data, x_column, y_column, title):
        fig = go.Figure()
        for x_value in sorted(data[x_column].unique()):
            subset = data[data[x_column] == x_value]
            fig.add_trace(go.Box(
                x=[str(x_value)] * len(subset),
                y=subset[y_column],
                name=str(x_value),
                boxpoints='all'
            ))
        fig.update_layout(
            title=title,
            xaxis_title=x_column,
            yaxis_title=y_column,
            height=600,
            width=1000
        )
        return fig

    # Taxonomy effects
    taxonomy_df = df[df['Typo Type'] == 'none']
    for metric in metrics:
        # Box plot for taxonomy effects
        fig = create_box_plot(
            taxonomy_df, 
            'Taxonomy Type', 
            metric, 
            f'{metric} by Taxonomy Type (Model: {model_name}, Strategy: {strategy})'
        )
        plot_file = os.path.join(plots_dir, f"plot_taxonomy_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"{metric} by Taxonomy Type"))

        # Bar plot for mean taxonomy effects
        mean_values = taxonomy_df.groupby('Taxonomy Type')[metric].mean().reset_index()
        fig = go.Figure(data=[
            go.Bar(x=mean_values['Taxonomy Type'], y=mean_values[metric])
        ])
        fig.update_layout(
            title=f'Mean {metric} by Taxonomy Type (Model: {model_name}, Strategy: {strategy})',
            xaxis_title='Taxonomy Type',
            yaxis_title=f'Mean {metric}',
            height=600,
            width=1000
        )
        plot_file = os.path.join(plots_dir, f"plot_mean_taxonomy_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f'Mean {metric} by Taxonomy Type'))

    # Typo effects
    typo_df = df[df['Taxonomy Type'] == '0']
    for metric in metrics:
        # Box plot for typo effects
        fig = make_subplots(rows=1, cols=2, subplot_titles=('Typo Type', 'Typo Intensity'))
        
        fig.add_trace(
            create_box_plot(
                typo_df, 
                'Typo Type', 
                metric, 
                ''
            ).data[0],
            row=1, col=1
        )
        
        fig.add_trace(
            create_box_plot(
                typo_df, 
                'Typo Intensity', 
                metric, 
                ''
            ).data[0],
            row=1, col=2
        )
        
        fig.update_layout(
            title=f'{metric} by Typo Type and Intensity (Model: {model_name}, Strategy: {strategy})',
            height=600,
            width=1000
        )
        plot_file = os.path.join(plots_dir, f"plot_typo_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"{metric} by Typo Type and Intensity"))

        # Heatmap for typo effects
        pivot_df = typo_df.pivot_table(values=metric, index='Typo Type', columns='Typo Intensity', aggfunc='mean')
        fig = go.Figure(data=go.Heatmap(
            z=pivot_df.values,
            x=pivot_df.columns,
            y=pivot_df.index,
            colorscale='Viridis'
        ))
        fig.update_layout(
            title=f'Mean {metric} by Typo Type and Intensity (Model: {model_name}, Strategy: {strategy})',
            xaxis_title='Typo Intensity',
            yaxis_title='Typo Type',
            height=600,
            width=1000
        )
        plot_file = os.path.join(plots_dir, f"plot_heatmap_typo_{metric.lower()}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f'Heatmap of Mean {metric} by Typo Type and Intensity'))

    return plots

def generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Taxonomy and Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Model: {model_name}", ln=True, align='C')
    pdf.cell(0, 10, f"Strategy: {strategy}", ln=True, align='C')
    pdf.cell(0, 10, f"Max New Tokens: {max_new_tokens}", ln=True, align='C')
    pdf.cell(0, 10, f"Temperature: {temperature}", ln=True, align='C')

    # Add plots and descriptions to the PDF
    for plot_file, desc in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.multi_cell(0, 10, desc)
        pdf.ln(5)
        pdf.image(plot_file, w=pdf.w - 20)
        
        # Add some interpretation text (you may want to customize this based on the actual results)
        pdf.ln(10)
        pdf.set_font("Arial", size=10)
        if "Taxonomy" in desc:
            pdf.multi_cell(0, 5, "This plot shows how different taxonomy types affect the model's performance. " 
                                 "Higher values indicate better performance. Variations across taxonomy types " 
                                 "may suggest areas where the model is more or less reliable.")
        elif "Typo" in desc:
            pdf.multi_cell(0, 5, "This plot illustrates the impact of different typo types and intensities on " 
                                 "the model's performance. Lower scores for certain typo types or higher intensities " 
                                 "indicate areas where the model's reliability decreases.")

    pdf.output(pdf_path, "F")

def main(exp_id, model_name, strategy, max_new_tokens, temperature):
    # Load the Excel file
    file_path = f"results/reliability_eval/unified_scores_table_{exp_id}.xlsx"
    df = pd.read_excel(file_path)

    # Filter the dataframe to include only the specified model and parameters
    df = df[(df['Model Name'] == model_name) & 
            (df['Strategy'] == strategy) & 
            (df['Max New Tokens'] == max_new_tokens) & 
            (df['Temperature'] == temperature)]

    # Create a directory for saving plots
    plots_dir = f"plots/taxonomy_typo_eval_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)

    # Generate and save the figures
    plots = create_and_save_figures(df, plots_dir, model_name, strategy, max_new_tokens, temperature)

    # Generate the PDF report
    pdf_path = os.path.join(plots_dir, f"taxonomy_typo_evaluation_plots_{exp_id}.pdf")
    generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature)

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Example usage
    main(
        exp_id="your-experiment-id",
        model_name="Your-Model-Name",
        strategy="Your-Strategy",
        max_new_tokens=25,
        temperature=0.1
    )